# Phase C2: Retrain with Pseudo-labels
Train EfficientNet-B0 + SED on clean audio mixed with pseudo-labeled soundscapes.
Changes from C1: MixUp ratio 1.0, Stochastic Depth 0.15, 30 epochs, patience 5.

In [8]:
# Cell 1: Config + Imports

import os
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'seed': 42,
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'pseudo_path': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/pseudo_labels.csv',
    'output_dir': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage2_sed',
    'sr': 32000,
    'duration': 20,
    'n_fft': 4096,
    'hop_length': 1252,
    'n_mels': 224,
    'fmin': 0,
    'fmax': 16000,
    'top_db': 80.0,
    'img_width': 512,
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    'num_classes': 234,
    'batch_size': 180,
    'epochs': 30,
    'patience': 5,
    'lr': 5e-4,
    'weight_decay': 1e-4,
    'mixup_weight': 0.5,
    'drop_path_rate': 0.15,
    'n_folds': 5,
    'num_workers': 4,
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")

Device: cuda (NVIDIA A40)


In [9]:
# Cell 2: Load all data sources

train = pd.read_csv(os.path.join(CFG['base_dir'], 'train.csv'))
taxonomy = pd.read_csv(os.path.join(CFG['base_dir'], 'taxonomy.csv'))

LABELS = sorted(taxonomy['primary_label'].tolist())
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

train['filepath'] = train['filename'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_audio', x)
)

# Pseudo-labels from C2 Step 1
pseudo_df = pd.read_csv(CFG['pseudo_path'])
pseudo_df['filepath'] = pseudo_df['soundscape_id'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_soundscapes', x + '.ogg')
)

# Labeled soundscapes (66 files, 1,478 rows)
sc_labels = pd.read_csv(os.path.join(CFG['base_dir'], 'train_soundscapes_labels.csv'))

# Parse start time from HH:MM:SS to seconds
def parse_time(t):
    parts = t.split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

sc_labels['start_sec'] = sc_labels['start'].apply(parse_time)
sc_labels['soundscape_id'] = sc_labels['filename'].str.replace('.ogg', '', regex=False)
sc_labels['filepath'] = sc_labels['filename'].apply(
    lambda x: os.path.join(CFG['base_dir'], 'train_soundscapes', x)
)

# Build label vectors for labeled soundscapes
sc_label_vectors = []
for _, row in sc_labels.iterrows():
    vec = np.zeros(CFG['num_classes'], dtype=np.float32)
    for species in str(row['primary_label']).split(';'):
        species = species.strip()
        if species in LABEL2IDX:
            vec[LABEL2IDX[species]] = 1.0
    sc_label_vectors.append(vec)
sc_labels['label_vector'] = sc_label_vectors

print(f"Clean training samples: {len(train):,}")
print(f"Pseudo-labeled segments: {len(pseudo_df):,} ({pseudo_df['soundscape_id'].nunique()} soundscapes)")
print(f"Labeled soundscape segments: {len(sc_labels):,} ({sc_labels['filename'].nunique()} soundscapes)")
print(f"Target species: {len(LABELS)}")

Clean training samples: 35,549
Pseudo-labeled segments: 127,104 (10592 soundscapes)
Labeled soundscape segments: 1,478 (66 soundscapes)
Target species: 234


In [10]:
# Cell 3: BirdDataset -- identical to C1 (for clean clips)

class BirdDataset(Dataset):
    def __init__(self, df, label2idx, cfg, is_train=True):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.cfg = cfg
        self.is_train = is_train
        self.target_len = cfg['sr'] * cfg['duration']

    def __len__(self):
        return len(self.df)

    def load_audio(self, filepath):
        try:
            y, sr = librosa.load(filepath, sr=self.cfg['sr'])
        except Exception:
            return np.zeros(self.target_len, dtype=np.float32)
        if len(y) == 0:
            return np.zeros(self.target_len, dtype=np.float32)
        if len(y) < self.target_len:
            reps = (self.target_len // len(y)) + 1
            y = np.tile(y, reps)
        if self.is_train and len(y) > self.target_len:
            start = random.randint(0, len(y) - self.target_len)
            y = y[start:start + self.target_len]
        else:
            center = len(y) // 2
            half = self.target_len // 2
            y = y[center - half:center - half + self.target_len]
        max_val = np.abs(y).max()
        if max_val > 0:
            y = y / max_val
        return y.astype(np.float32)

    def audio_to_melspec(self, y):
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.cfg['sr'],
            n_fft=self.cfg['n_fft'], hop_length=self.cfg['hop_length'],
            n_mels=self.cfg['n_mels'], fmin=self.cfg['fmin'], fmax=self.cfg['fmax'],
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=self.cfg['top_db'])
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = self.load_audio(row['filepath'])
        mel = self.audio_to_melspec(y)
        if mel.shape[1] < self.cfg['img_width']:
            mel = np.pad(mel, ((0, 0), (0, self.cfg['img_width'] - mel.shape[1])))
        else:
            mel = mel[:, :self.cfg['img_width']]
        mel = np.stack([mel, mel, mel], axis=0)
        mel = torch.tensor(mel, dtype=torch.float32)
        target = torch.zeros(self.cfg['num_classes'], dtype=torch.float32)
        if row['primary_label'] in self.label2idx:
            target[self.label2idx[row['primary_label']]] = 1.0
        return mel, target

print("BirdDataset defined (clean clips).")

BirdDataset defined (clean clips).


In [11]:
# Cell 4: SoundscapeDataset -- loads 20-sec chunk from soundscape at given start_sec
# Used for both pseudo-labeled (soft labels) and labeled soundscapes (hard labels)

class SoundscapeDataset(Dataset):
    def __init__(self, df, labels_list, cfg):
        """
        df: DataFrame with columns 'filepath', 'start_sec'
        labels_list: list of label vectors (np arrays of shape [num_classes])
        """
        self.df = df.reset_index(drop=True)
        self.labels = labels_list
        self.cfg = cfg
        self.target_len = cfg['sr'] * cfg['duration']
        self._audio_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_full_audio(self, filepath):
        if filepath not in self._audio_cache:
            try:
                y, _ = librosa.load(filepath, sr=self.cfg['sr'])
            except Exception:
                y = np.zeros(self.cfg['sr'] * 60, dtype=np.float32)
            self._audio_cache[filepath] = y
            # Keep cache bounded
            if len(self._audio_cache) > 200:
                oldest = next(iter(self._audio_cache))
                del self._audio_cache[oldest]
        return self._audio_cache[filepath]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_audio = self._load_full_audio(row['filepath'])
        start_sample = int(row['start_sec']) * self.cfg['sr']

        # Extract 20-sec chunk centered on the 5-sec segment
        # Center the window: segment is 5s, window is 20s, so offset by 7.5s before
        center_sample = start_sample + (self.cfg['sr'] * 5 // 2)
        half_window = self.target_len // 2
        chunk_start = max(0, center_sample - half_window)
        chunk_end = chunk_start + self.target_len

        if chunk_end > len(full_audio):
            chunk_start = max(0, len(full_audio) - self.target_len)
            chunk_end = chunk_start + self.target_len

        y = full_audio[chunk_start:chunk_end]

        # Pad if audio is shorter than window
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))

        # Absmax normalization
        max_val = np.abs(y).max()
        if max_val > 0:
            y = y / max_val

        # Mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=y.astype(np.float32), sr=self.cfg['sr'],
            n_fft=self.cfg['n_fft'], hop_length=self.cfg['hop_length'],
            n_mels=self.cfg['n_mels'], fmin=self.cfg['fmin'], fmax=self.cfg['fmax'],
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=self.cfg['top_db'])
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

        if mel_db.shape[1] < self.cfg['img_width']:
            mel_db = np.pad(mel_db, ((0, 0), (0, self.cfg['img_width'] - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :self.cfg['img_width']]

        mel_3ch = np.stack([mel_db, mel_db, mel_db], axis=0)
        mel_tensor = torch.tensor(mel_3ch, dtype=torch.float32)
        target = torch.tensor(self.labels[idx], dtype=torch.float32)

        return mel_tensor, target

print("SoundscapeDataset defined.")

SoundscapeDataset defined.


In [12]:
# Cell 5: GeM + SEDModel (with Stochastic Depth)

class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=-1).pow(1.0 / self.p)


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['model_name'], pretrained=True,
            in_chans=3, num_classes=0, global_pool='',
            drop_path_rate=cfg.get('drop_path_rate', 0.0),
        )
        self.feat_dim = 1280
        self.freq_pool = GeM(p=3)
        self.fc = nn.Linear(self.feat_dim, cfg['num_classes'])

    def forward(self, x):
        features = self.backbone(x)
        x = self.freq_pool(features)
        x = x.transpose(1, 2)
        frame_logits = self.fc(x)
        clip_logits = frame_logits.mean(dim=1)
        clip_logits_max = frame_logits.max(dim=1)[0]
        output = 0.5 * clip_logits + 0.5 * clip_logits_max
        return output

# Quick test
model = SEDModel(CFG).to(DEVICE)
dummy = torch.randn(2, 3, 224, 512).to(DEVICE)
out = model(dummy)
print(f"Output shape: {out.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Stochastic Depth: drop_path_rate={CFG['drop_path_rate']}")
del model, dummy, out
torch.cuda.empty_cache()

Output shape: torch.Size([2, 234])
Total parameters: 4,307,303
Stochastic Depth: drop_path_rate=0.15


In [7]:
# Max Batch size

# Batch size test for C2 training (with gradients + Stochastic Depth)
import gc

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()
gc.collect()

model = SEDModel(CFG).to(DEVICE)
model.train()

for bs in [32, 64, 128, 180, 200, 212]:
    try:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        
        dummy = torch.randn(bs, 3, 224, 512).to(DEVICE)
        out = model(dummy)
        loss = out.sum()
        loss.backward()
        
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"BS {bs}: OK (peak GPU: {peak:.1f} GB / 48 GB)")
        
        model.zero_grad()
        del dummy, out, loss
        torch.cuda.empty_cache()
    except RuntimeError:
        print(f"BS {bs}: OOM")
        torch.cuda.empty_cache()
        gc.collect()
        break

del model
torch.cuda.empty_cache()

BS 32: OK (peak GPU: 6.7 GB / 48 GB)
BS 64: OK (peak GPU: 13.5 GB / 48 GB)
BS 128: OK (peak GPU: 26.9 GB / 48 GB)
BS 180: OK (peak GPU: 37.7 GB / 48 GB)
BS 200: OK (peak GPU: 41.9 GB / 48 GB)
BS 212: OK (peak GPU: 44.4 GB / 48 GB)


In [13]:
# Cell 6: MixUp + SpecAugment (same as C1)

def mixup(mel, target, mel2, target2, weight=0.5):
    """MixUp: blend clean sample with pseudo-labeled sample.
    Target: element-wise max (from 2025 1st place).
    """
    mel_mixed = weight * mel + (1 - weight) * mel2
    target_mixed = torch.max(target, target2)
    return mel_mixed, target_mixed


def spec_augment(mel, freq_mask=30, time_mask=50):
    _, n_mels, n_time = mel.shape
    f = random.randint(0, freq_mask)
    f0 = random.randint(0, max(0, n_mels - f))
    mel[:, f0:f0+f, :] = 0
    t = random.randint(0, time_mask)
    t0 = random.randint(0, max(0, n_time - t))
    mel[:, :, t0:t0+t] = 0
    return mel

print("MixUp (cross-source, ratio 1.0) and SpecAugment defined.")

MixUp (cross-source, ratio 1.0) and SpecAugment defined.


In [14]:
# Cell 7: Training function -- MixUp ratio 1.0 (every batch mixed with pseudo-labeled data)

def train_one_epoch_c2(model, clean_loader, pseudo_loader, optimizer, scheduler, device, cfg):
    model.train()
    losses = []
    criterion = nn.CrossEntropyLoss()
    pseudo_iter = iter(pseudo_loader)

    pbar = tqdm(clean_loader, desc="Train")
    for mel_clean, target_clean in pbar:
        mel_clean = mel_clean.to(device)
        target_clean = target_clean.to(device)

        # Get pseudo-labeled batch (cycle if exhausted)
        try:
            mel_pseudo, target_pseudo = next(pseudo_iter)
        except StopIteration:
            pseudo_iter = iter(pseudo_loader)
            mel_pseudo, target_pseudo = next(pseudo_iter)

        mel_pseudo = mel_pseudo.to(device)
        target_pseudo = target_pseudo.to(device)

        # Match batch sizes (trim larger to smaller)
        bs = min(mel_clean.size(0), mel_pseudo.size(0))
        mel_clean, target_clean = mel_clean[:bs], target_clean[:bs]
        mel_pseudo, target_pseudo = mel_pseudo[:bs], target_pseudo[:bs]

        # MixUp ratio 1.0: EVERY clean sample mixed with a pseudo sample
        mel, target = mixup(mel_clean, target_clean, mel_pseudo, target_pseudo,
                            weight=cfg['mixup_weight'])

        # SpecAugment
        for i in range(mel.size(0)):
            if random.random() < 0.5:
                mel[i] = spec_augment(mel[i])

        logits = model(mel)
        loss = criterion(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")

    scheduler.step()
    return np.mean(losses)

print("train_one_epoch_c2 defined.")

train_one_epoch_c2 defined.


In [15]:
# Cell 8: Validation (identical to C1)

def validate(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for mel, target in tqdm(loader, desc="Valid"):
            mel, target = mel.to(device), target.to(device)
            logits = model(mel)
            loss = criterion(logits, target)
            losses.append(loss.item())
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    aucs = []
    for i in range(all_targets.shape[1]):
        if all_targets[:, i].sum() > 0:
            try:
                auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
                aucs.append(auc)
            except ValueError:
                pass

    macro_auc = np.mean(aucs) if aucs else 0.0
    return np.mean(losses), macro_auc, len(aucs)

print("validate defined.")

validate defined.


In [16]:
# Cell 9: 5-fold training loop

# Build combined pseudo + labeled soundscape dataset
# Pseudo-labels: soft probability vectors from pseudo_labels.csv
pseudo_label_vectors = pseudo_df[LABELS].values.astype(np.float32)

# Labeled soundscapes: hard label vectors
sc_label_vecs = np.stack(sc_labels['label_vector'].values).astype(np.float32)

# Combine pseudo + labeled into one soundscape pool
soundscape_pool_df = pd.concat([
    pseudo_df[['filepath', 'start_sec']],
    sc_labels[['filepath', 'start_sec']],
], ignore_index=True)
soundscape_pool_labels = np.concatenate([pseudo_label_vectors, sc_label_vecs], axis=0)

print(f"Soundscape pool: {len(soundscape_pool_df):,} segments")
print(f"  Pseudo-labeled: {len(pseudo_df):,}")
print(f"  Labeled: {len(sc_labels):,}")

# Same fold splits as C1 (same seed, same StratifiedKFold on clean data)
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['primary_label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    train_df = train.iloc[train_idx]
    val_df = train.iloc[val_idx]
    print(f"Clean train: {len(train_df):,} | Val: {len(val_df):,}")

    # Clean data loaders
    train_ds = BirdDataset(train_df, LABEL2IDX, CFG, is_train=True)
    val_ds = BirdDataset(val_df, LABEL2IDX, CFG, is_train=False)

    clean_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)

    # Soundscape pseudo-label loader (shuffled, cycles infinitely via train loop)
    soundscape_ds = SoundscapeDataset(
        soundscape_pool_df,
        [soundscape_pool_labels[i] for i in range(len(soundscape_pool_labels))],
        CFG,
    )
    pseudo_loader = DataLoader(soundscape_ds, batch_size=CFG['batch_size'], shuffle=True,
                               num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)

    print(f"Soundscape pool: {len(soundscape_ds):,} segments")

    model = SEDModel(CFG).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, eta_min=1e-6)

    best_auc = 0
    no_improve = 0

    for epoch in range(CFG['epochs']):
        print(f"\n--- Epoch {epoch+1}/{CFG['epochs']} ---")

        train_loss = train_one_epoch_c2(model, clean_loader, pseudo_loader,
                                         optimizer, scheduler, DEVICE, CFG)
        val_loss, val_auc, n_scored = validate(model, val_loader, DEVICE)

        print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} (on {n_scored} species)")

        if val_auc > best_auc:
            best_auc = val_auc
            no_improve = 0
            save_path = os.path.join(CFG['output_dir'], f'c2_fold{fold}.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best model (AUC={best_auc:.4f})")
        else:
            no_improve += 1
            print(f"  -> No improvement ({no_improve}/{CFG['patience']})")
            if no_improve >= CFG['patience']:
                print(f"  -> Early stopping at epoch {epoch+1}")
                break

    fold_results.append({'fold': fold, 'best_auc': best_auc})
    print(f"\nFold {fold} best AUC: {best_auc:.4f}")

    del model, optimizer, scheduler, clean_loader, val_loader, pseudo_loader
    del train_ds, val_ds, soundscape_ds
    torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print(f"C2 CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(results_df.to_string(index=False))
print(f"\nMean AUC: {results_df['best_auc'].mean():.4f} +/- {results_df['best_auc'].std():.4f}")

Soundscape pool: 128,582 segments
  Pseudo-labeled: 127,104
  Labeled: 1,478

FOLD 0
Clean train: 28,439 | Val: 7,110
Soundscape pool: 128,582 segments

--- Epoch 1/30 ---


Valid: 100%|██████████| 40/40 [02:36<00:00,  3.91s/it]


Train loss: 478.8290 | Val loss: 4.9863 | Val AUC: 0.7335 (on 198 species)
  -> Saved best model (AUC=0.7335)

--- Epoch 2/30 ---


Valid: 100%|██████████| 40/40 [02:37<00:00,  3.95s/it]


Train loss: 476.3504 | Val loss: 4.9324 | Val AUC: 0.7691 (on 198 species)
  -> Saved best model (AUC=0.7691)

--- Epoch 3/30 ---


...

Train loss: 475.9989 | Val loss: 4.9195 | Val AUC: 0.7904 (on 199 species)
  -> Saved best model (AUC=0.7904)

--- Epoch 4/30 ---


Valid: 100%|██████████| 40/40 [02:32<00:00,  3.82s/it]


Train loss: 474.6756 | Val loss: 4.9145 | Val AUC: 0.7938 (on 199 species)
  -> Saved best model (AUC=0.7938)

--- Epoch 5/30 ---


Train:  39%|███▉      | 62/157 [10:18<15:47,  9.97s/it, loss=475.5420] 


KeyboardInterrupt: 

In [ ]:
# Cell 10: Sample predictions with best C2 model

model = SEDModel(CFG).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(CFG['output_dir'], 'c2_fold0.pth'), map_location=DEVICE))
model.eval()

val_df = train.iloc[list(skf.split(train, train['primary_label']))[0][1]]
samples = val_df.sample(5, random_state=42)
val_ds = BirdDataset(samples, LABEL2IDX, CFG, is_train=False)

print("Sample predictions (top 5 species per clip):\n")
for i in range(len(val_ds)):
    mel, target = val_ds[i]
    row = samples.iloc[i]
    with torch.no_grad():
        logits = model(mel.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    true_label = row['primary_label']
    true_name = taxonomy.loc[taxonomy['primary_label'] == true_label, 'common_name'].values
    true_name = true_name[0] if len(true_name) > 0 else '?'

    top5_idx = probs.argsort()[-5:][::-1]
    print(f"True: {true_label} ({true_name})")
    for idx in top5_idx:
        marker = ' <-- CORRECT' if IDX2LABEL[idx] == true_label else ''
        print(f"    {IDX2LABEL[idx]:>12s}: {probs[idx]:.4f}{marker}")
    print()

print("C2 complete!")